In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import XGBRegressor
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import math
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
# Load dataset

data = pd.read_csv('Stock Market.csv')
data

In [ ]:
# Converting date back to string

data['Date'] = data['Date'].apply(lambda x: ''.join([c for c in x if isinstance(c, str)]))

In [ ]:
# Reconstruction of date

data['Date'] = data['Date'].str.replace(r"(\\d{2})(\\d{2})(\\d{4})", r"\\1-\\2-\\3", regex=True)

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
# checking data info

data.info()

In [ ]:
data.describe()

In [ ]:
data.nunique()

In [ ]:
data.head(10)

In [ ]:
data.tail()

In [ ]:
# Checking Missing Values

data.isna().sum()

In [ ]:
# Fill missing numeric values with previous day (forward fill)

data.fillna(method='ffill', inplace=True)
data

In [ ]:
# Checking for duplicates

data.duplicated().sum()

In [ ]:
# Removing Duplicates

data.drop_duplicates(inplace=True)

In [ ]:
# Checking the Datatypes

data.dtypes

In [ ]:
# Changing the Datatype

data['Date'] = pd.to_datetime(data['Date'],format='%d-%m-%Y',errors='raise')

In [ ]:
#Sorting the Date

data = data.sort_values('Date').reset_index(drop=True)
data

In [ ]:
# datatypes after changing date datatype

data.dtypes

In [ ]:
# Dataset

data.head()

# Feature Engineering:

## Moving averages & volatility

In [ ]:
# Creating Moving Averages and Volatility

df = data[['Date','Close']].copy()
df['ma7'] = df['Close'].rolling(7).mean()
df['ma14'] = df['Close'].rolling(14).mean()
df['ma21'] = df['Close'].rolling(21).mean()
df['volatility'] = df['Close'].rolling(7).std()  # 7-day rolling volatility
df = df.dropna()
df.set_index('Date', inplace=True)
df

## Time features

In [ ]:
# Creating time features

df['day_of_week'] = df.index.dayofweek
df['week_of_year'] = df.index.isocalendar().week
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['year'] = df.index.year
df

In [ ]:
for lag in [1,2,3,5,7]:
    df[f'lag_{lag}'] = df['Close'].shift(lag)  # Lag features provide the previous days’ closing prices.


In [ ]:
df['SP500_Close'] = df['Close'] * 1.02  # Approximation of overall market trend. Stocks often move with the market.
df['Inflation'] = 2.1  # Macro-economic factor
df['Earnings_Season'] = df.index.month.isin([1,4,7,10]).astype(int)  # Captures quarterly earnings announcements (Jan, Apr, Jul, Oct).
df['Target'] = df['Close'].shift(-1)   #Next-day closing price (what we want to predict)

In [ ]:
df.dropna(inplace=True)  # Remove rows with missing values created by rolling/lag
df.reset_index(drop=False, inplace=True)  # Dropping these ensures clean data for modeling.

# Explanatory Data Analysis(EDA)

# Line Plot

In [ ]:
# Plotting Closing prices 
plt.figure(figsize=(9,6))
plt.title('CLOSE PRICE OF APPLE STOCK FROM 2012 TO 2019',fontsize=18,color='#8207DB')
plt.xlabel('Date')
plt.ylabel('Close')

#Line Plot
plt.plot(data['Close'],label='Close',color='red')
plt.legend()
plt.show()

In [ ]:
decompose_result = seasonal_decompose(df['Close'], model='multiplicative', period=252)
decompose_result.plot()
plt.show()

In [ ]:
# Plotting Volume Trend 
plt.figure(figsize=(9,6))
plt.title('VOLUME OF APPLE STOCK FROM 2012 TO 2019',fontsize=18,color='#8207DB')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.plot(data['Volume'],label='Volume',color='brown')
plt.legend()
plt.show()

In [ ]:
# Line plot Moving Averages
plt.figure(figsize=(9,6))
plt.plot(df.index, df['Close'], label='Close',color='red')
plt.plot(df.index, df['ma7'], label='MA7',color = 'black')
plt.plot(df.index, df['ma14'], label='MA14',color='purple')
plt.plot(df.index, df['ma21'], label='MA21')
plt.legend()
plt.title("STOCK PRICES AND MOVING AVERAGES", color='blue')
plt.show()

# Histogram

In [ ]:

plt.figure(figsize=(10,6))
sns.histplot(df['Close'],kde=True,bins=30)
plt.title("Distribution of Closing Prices")
plt.xlabel("Close Price")
plt.show()

In [ ]:
# Histogram
df[['Close','ma7','ma14','ma21','volatility']].hist(figsize=(10,6), bins=30)
plt.show()

# Boxplot

In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=data['Close'],
            boxprops = dict(facecolor ='green', edgecolor='midnightblue')
           )
plt.title("BOXPLOT OF CLOSE PRICE")
plt.show()

# Correlation heatmap

In [ ]:
corr = df.corr(numeric_only=True)
corr

In [ ]:
# Correlated heatmap
plt.figure(figsize=(15,10))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("CORRELATION HEATMAP")
plt.show()

# Pair Plot

In [ ]:
# Pairplot
sns.pairplot(data[['Open','High','Low','Close','Adj Close','Volume']].sample(300))
plt.show()

# Train-Test Split

In [ ]:
X = df.select_dtypes(include=['number']).copy()
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# XGBoost Model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Training XGBoost Model and Predicting Using XGBoost Model

xgb_model = xgb.XGBRegressor(objective='reg:squarederror',n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_pred

In [ ]:
# Evaluate XGBoost

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = math.sqrt(mean_squared_error(y_test, xgb_pred))

print("XGB MAE:", xgb_mae)
print("XGB RMSE:", xgb_rmse)

# ARIMA Model

In [ ]:
# ARIMA works only on 1D series

train_arima = df['Close'][:-30]
test_arima = df['Close'][-30:]

# Fit ARIMA Model

arima_model = ARIMA(train_arima, order=(5,1,2))
arima_result = arima_model.fit()
arima_result.summary()

In [ ]:
# Predicting using ARIMA Model
arima_pred = arima_result.forecast(steps=len(y_test))
arima_pred

In [ ]:
# Evaluate ARIMA

arima_mae = mean_absolute_error(y_test, arima_pred)
arima_rmse = math.sqrt(mean_squared_error(y_test, arima_pred))

print("ARIMA MAE:", arima_mae)
print("ARIMA RMSE:", arima_rmse)

# SARIMA Model

In [ ]:
# Fit SARIMA Model

train_sarima = df['Close'][:-30]
test_sarima = df['Close'][-30:]

sarima_model = SARIMAX(train_sarima, order=(2,1,2), seasonal_order=(1,1,1,12))
sarima_result = sarima_model.fit()
sarima_result.summary()

In [ ]:
# Predict with SARIMA

sarima_pred = sarima_result.forecast(steps=len(y_test))
sarima_pred

In [ ]:
# SARIMA Evaluation

sarima_mae = mean_absolute_error(y_test, sarima_pred)
sarima_rmse = math.sqrt(mean_squared_error(y_test, sarima_pred))

print("SARIMA MAE:", sarima_mae)
print("SARIMA RMSE:", sarima_rmse)

# Plotting the Evaluation 

In [ ]:
plt.figure(figsize=(9,6))
plt.plot(y_test.index, y_test, label='Actual')
plt.plot(y_test.index, xgb_pred, label='XGBoost')
plt.plot(y_test.index, arima_pred, label='ARIMA')
plt.plot(y_test.index, sarima_pred, label='SARIMA')
plt.legend()
plt.title('Actual Vs Predictions')
plt.show()

# Forecasting the next 30 Business Days(Using XGBoost Model)

In [ ]:
# Create next 30 predictions(XGBoost Model)

temp_df = df.copy()
temp_df.index = pd.to_datetime(temp_df.index)

future_dates =  pd.date_range(start=df['Date'].iloc[-1]+pd.Timedelta(days=1), periods=30)
last_window = data.iloc[-14:].copy()

future_preds_xgb = []

for d in future_dates:
    current_close =last_window['Close'].iloc[-1]
    lag1 = last_window['Close'].iloc[-1]
    lag2 = last_window['Close'].iloc[-2]
    lag3 = last_window['Close'].iloc[-3]
    lag5 = last_window['Close'].iloc[-5]
    lag7 = last_window['Close'].iloc[-7]
    ma7 = last_window['Close'].rolling(7).mean().iloc[-1]
    ma14 = last_window['Close'].rolling(14).mean().iloc[-1]
    ma21 = last_window['Close'].rolling(21).mean().iloc[-1]
    vol = last_window['Close'].rolling(7).std().iloc[-1]
    dow = d.dayofweek
    week = d.isocalendar().week
    month = d.month
    quarter = d.quarter
    year = d.year
    sp500_close = lag1 * 1.02
    inflation = 2.1
    earnings_season = int(month in [1, 4, 7, 10])

    X_new = pd.DataFrame([{
        'Close': current_close,
        'ma7': ma7,
        'ma14': ma14,
        'ma21': ma21,
        'volatility': vol,
        'day_of_week': dow,
        'week_of_year': int(week),
        'month': month,
        'quarter': quarter,
        'year': year,
        'lag_1': lag1,
        'lag_2': lag2,
        'lag_3': lag3,
        'lag_5': lag5,
        'lag_7': lag7,
        'SP500_Close': sp500_close,
        'Inflation': inflation,
        'Earnings_Season': earnings_season,
        'Target': 0          # ✅ dummy placeholder (REQUIRED)
    }])
    pred_val = xgb_model.predict(X_new)[0]

    future_preds_xgb.append((d, pred_val))

    new = pd.DataFrame({'Close': pred_val}, index=[d])
    last_window = pd.concat([last_window, new])
    last_window = last_window.tail(14)


In [ ]:
# Convert forecast to DataFrame

forecast_df_xgb = pd.DataFrame(future_preds_xgb, columns=['Date','Predicted'])
forecast_df_xgb.set_index('Date', inplace=True)
forecast_df_xgb

In [ ]:
# Plotting the Forecast

plt.figure(figsize=(10,6))
plt.plot(df.index, df['Close'], label='History')
plt.plot(forecast_df_xgb.index, forecast_df_xgb['Predicted'], marker='o', label='Forecast')
plt.legend()
plt.title("30-Day Forecast (XGBoost)")
plt.show()

# Saving forecast as CSV

In [ ]:
# Save forecast as CSV

final_forecast = pd.DataFrame()

final_forecast['Date'] = forecast_df_xgb.index
final_forecast['XGB_Prediction'] = forecast_df_xgb['Predicted'].values

final_forecast.to_csv("XGBoost_forecast.csv", index=False)

final_forecast

# Deployment with Streamlit:

In [ ]:
import pickle 

In [ ]:
from pickle import dump

In [ ]:
# Save the model

pickle.dump(xgb_model,open('xgb_model.pkl','wb'))

In [ ]:
# Loading the model

loaded_model = pickle.load(open('xgb_model.pkl','rb'))